In [1]:
import os
os.chdir('/workspace/13b46120-8b00-4771-a2a8-e33ae13d7631')
print(os.listdir('.'))


['F16_perturbation_results.csv', 'imc3.npy', 'rec3.npy', ' v6 — Binding Context Document.pdf', '.kernel_llm_logs_1.txt', 're15.npy', 'im15.npy', '.config', 'memory', '.prompts', 'ts.npy']


In [2]:
import numpy as np
print('files:')
for f in ['ts.npy', 're15.npy', 'im15.npy', 'rec3.npy', 'imc3.npy']:
 a = np.load(f, mmap_mode='r')
 print(f, a.shape, a.dtype)


files:
ts.npy (100001,) float64
re15.npy (100001,) float64
im15.npy (100001,) float64
rec3.npy (100001,) float64
imc3.npy (100001,) float64


In [3]:
import pandas as pd
df = pd.read_csv('F16_perturbation_results.csv')
print(df)


 c pc1_complex H_complex trace
0 0.00 0.940518 0.259996 36.217954
1 0.20 0.935020 0.290158 37.031201
2 0.40 0.920611 0.352641 39.784791
3 0.45 0.917837 0.364201 40.688674
4 0.50 0.914962 0.374457 41.978616
5 0.55 0.913306 0.383684 43.167684
6 0.60 0.909387 0.401305 44.408274
7 0.70 0.905375 0.419119 47.846557
8 0.80 0.899796 0.442141 52.473462
9 0.90 0.900214 0.442566 57.721963
10 1.00 0.905773 0.424298 63.751104


In [4]:
# Look for partial sums file referenced
import os
candidates = []
for root, dirs, files in os.walk('/workspace'):
 # skip deep dirs
 depth = root.count('/') - 2
 if depth > 4: 
 continue
 for f in files:
 if 'partial_sums_F15' in f or 'F16' in f or 'Sk_complex' in f:
 candidates.append(os.path.join(root, f))
print('\n'.join(candidates[:50]))


/workspace/13b46120-8b00-4771-a2a8-e33ae13d7631/F16_perturbation_results.csv


In [5]:
# Step 1: Load coarse-grid partial sums
import numpy as np

ts = np.load('ts.npy')
re15 = np.load('re15.npy')
im15 = np.load('im15.npy')
rec3 = np.load('rec3.npy')
imc3 = np.load('imc3.npy')

F15 = re15 + 1j*im15
chi3 = rec3 + 1j*imc3

print("t range:", ts[0], ts[-1], "n=", len(ts), "dt=", ts[1]-ts[0])
print("|F15| stats:", np.abs(F15).min(), np.abs(F15).max(), np.abs(F15).mean())
print("|chi3| stats:", np.abs(chi3).min(), np.abs(chi3).max(), np.abs(chi3).mean())


t range: 10000.0 20000.00000003638 n= 100001 dt= 0.1000000000003638
|F15| stats: 0.0027081360334364935 18.080472097690137 1.752476179211599
|chi3| stats: 0.00018270615508845492 16.028804699560585 1.8276315362629694


In [6]:
# Step 2: Identify top 200 peaks of |F16(c=0)| = |F15| and |F16(c=0.6)| = |F15 + 0.6*chi3|
from scipy.signal import find_peaks

def top_peaks(absF, ts, n=200):
 # Find local maxima
 peaks, _ = find_peaks(absF)
 # Top 200 by amplitude
 peak_vals = absF[peaks]
 top_idx = peaks[np.argsort(peak_vals)[-n:]]
 top_idx = np.sort(top_idx)
 return ts[top_idx], top_idx

absF0 = np.abs(F15 + 0.0*chi3)
absF06 = np.abs(F15 + 0.6*chi3)

t_peaks_0, idx0 = top_peaks(absF0, ts, 200)
t_peaks_06, idx06 = top_peaks(absF06, ts, 200)
print("c=0 peaks: n=", len(t_peaks_0), "range:", t_peaks_0.min(), t_peaks_0.max(), "min |F|:", absF0[idx0].min())
print("c=0.6 peaks: n=", len(t_peaks_06), "range:", t_peaks_06.min(), t_peaks_06.max(), "min |F|:", absF06[idx06].min())


c=0 peaks: n= 200 range: 10043.300000000158 19996.400000036367 min |F|: 9.223921006436512
c=0.6 peaks: n= 200 range: 10043.300000000158 19987.30000003633 min |F|: 10.436655668404537


In [7]:
# Step 3: Compute complex omega-stratified sums S_k(F15) and S_k(chi3) at c=0.6 peak locations.
# We need coefficients of F15 and chi3, stratified by omega(n) = number of distinct prime factors.
# F15 is described as a GRH-true function. Let's check memory for what F15 specifically is.


In [8]:
# We need to identify F15 more precisely — let's check if F15 corresponds to a Dirichlet L-function 
# whose first values match the partial sums at t=10000.
# At t=10000: re15 + i*im15. Compare to known L functions.
# Check zeta first
import mpmath as mp
mp.mp.dps = 30
print("zeta(0.5+10000i):", complex(mp.zeta(0.5+10000j)))
print("F15(t=10000):", F15[0])


zeta(0.5+10000i): (-0.33937380263883443-0.03709150597320603j)
F15(t=10000): (0.6472438001771406-0.42959535610136734j)


In [9]:
# Not zeta. F15 is probably not zeta. Test other candidates.
# Try L(s, chi4) (Dirichlet L for non-principal char mod 4) - GRH-true, non-trivial
def L_chi4(s):
 # chi4(n): 1, 0, -1, 0, repeating
 return mp.dirichlet(s, [1, 0, -1, 0])

print("L(0.5+10000i, chi4):", complex(L_chi4(0.5+10000j)))

# Try L(s, chi3) with chi3 = [0,1,-1,0,1,-1,...]
def L_chi3(s):
 return mp.dirichlet(s, [0, 1, -1])
print("L(0.5+10000i, chi3):", complex(L_chi3(0.5+10000j)))
print("chi3 from file:", chi3[0])


L(0.5+10000i, chi4): (0.30900804239168744+0.08539177117097699j)


L(0.5+10000i, chi3): (2.0740885621250746+2.6547138613697037j)
chi3 from file: (2.07469398206086+2.6544347028634334j)


In [10]:
# Good - L(s, chi3) matches with chi3(n) given by [0, 1, -1, 0, 1, -1,...] (non-principal mod 3, real).
# Difference small (partial sum N=10^6 vs full).
# Now identify F15. It's not zeta or chi4.
# Try modular L (e.g., L(s, Δ) - Ramanujan tau, F9 from binding context)
# tau coefficients - mpmath.cv? Let's try via known formula.
# Or try a Dirichlet character of higher modulus.
# Actually, simplest: the description says "GRH-true". Could it be L(s,χ_5) or L(s,Δ).
# Let's try a few more characters.

for q in [5, 7, 8, 11, 12]:
 for chi_idx in range(2, q): # try non-principal real ones
 # try a real quadratic character mod q
 pass

# Try chi_5 - non-principal character with values chi(1)=1, chi(2)=i, etc... too many.
# Try L(s, chi_-3) which is what chi3 already is. Let's try L(s, chi_-4) = chi4 — no match.
# Try L(s, chi_-7) (Kronecker -7)
def kronecker_chi(d, n):
 # Kronecker symbol (d/n) for d squarefree
 if n == 0: return 0
 from sympy import jacobi_symbol, gcd
 if gcd(n, d) != 1:
 return 0
 return jacobi_symbol(d, n) if d > 0 else jacobi_symbol(d % n, n) * (1 if (d > 0 or n % 4 != 3) else -1)

# Use mpmath for speed - they can take period.
# chi_-7: period 7, values [0, 1, 1, -1, 1, -1, -1]?
# Actually for primitive real char mod 7: values for n=1..7: 1,1,-1,1,-1,-1,0 (sum 0)
chars = {
 'chi_-7 (mod 7 real)': [0, 1, 1, -1, 1, -1, -1],
 'chi_-8': [0,1,0,1,0,-1,0,-1], # mod 8
 'chi_8 (8a)': [0,1,0,-1,0,-1,0,1],
 'chi_5_real': [0,1,-1,-1,1], # mod 5 real
 'chi_-11': [0,1,-1,1,1,1,-1,-1,-1,1,-1], # mod 11
}
for name, c in chars.items():
 val = complex(mp.dirichlet(0.5+10000j, c))
 print(f"{name}: {val}, |target-this|=", abs(val - F15[0]))
print("Target F15:", F15[0])


chi_-7 (mod 7 real): (6.2075750193456-5.387976835227869j), |target-this|= 7.450022158316317


chi_-8: (1.0306555262258552+1.0348702955629259j), |target-this|= 1.5138243612044886


chi_8 (8a): (-0.0026177773147508707+1.2828902857751574j), |target-this|= 1.8316460202597835


chi_5_real: (0.04402897559746904-0.4027745445304132j), |target-this|= 0.6038107986165936


chi_-11: (0.15277601738815286+0.5639854775989709j), |target-this|= 1.1098203734446925
Target F15: (0.6472438001771406-0.42959535610136734j)


In [11]:
# None match. F15 is probably a modular L-function. Let me try L(s, Δ) (Ramanujan delta).
# tau(n) values - first few: 1, -24, 252, -1472, 4830, -6048, ...
# But L(s, Δ) is normalized to critical strip 0<Re(s)<1 by shifting: L(s, Δ) uses tau(n)/n^(11/2).

# Actually in many papers normalize using analytic normalization a_n = tau(n)/n^(11/2)
# Let me check.
# The list of known F1-F13 includes F1 (zeta) and modular L-functions F9-F11.

# Could F15 be something like the Dedekind zeta of Q(sqrt(-3))? That's zeta * L(s,chi3).
# F15(t) at t=10000: 0.647 - 0.430i
# zeta * chi3: (-0.339 - 0.037i) * (2.074 + 2.655i) = ?
prod = complex(mp.zeta(0.5+10000j)) * chi3[0]
print("zeta*chi3:", prod)
# Hmm not it.

# Try Dedekind zeta-like: but that's not GRH-friendly.
# Try L(s, Δ) (analytically normalized)
import mpmath as mp
# mpmath has mp.taueta, etc., not built-in for delta easily.
# Let me try sympy or compute tau directly.

# Alternative: maybe F15 is the Hecke L-function of some specific cusp form. Without more info, 
# let me directly check if F15 matches some simple combination: e.g. zeta * something.

# Actually looking at the problem: F16(c) = F15 + c*L(s, chi3). We have explicit data of partial sums 
# of F15 and chi3. We don't need to identify F15 by name — we need its Dirichlet coefficients to 
# build S_k(F15) (omega-stratified sums).
# But we don't have F15 coefficients! We only have its partial-sum values on the t-grid.
# Without coefficients we cannot compute S_k.

# Step: determine if F15's coefficients can be recovered/identified.
print("\nLet's see if F15(0.5+it=10000) is consistent with a simple case.")
print("Ratio F15/chi3:", F15[0]/chi3[0])
# Try zeta*L(s,chi5_-)


zeta*chi3: (-0.6056398053671553-0.9777991231954329j)

Let's see if F15(0.5+it=10000) is consistent with a simple case.
Ratio F15/chi3: (0.01784081402796214-0.22989059403871723j)


In [12]:
# Let's try L(s, Δ) — Ramanujan delta L-function, analytically normalized.
# Use mpmath: not directly. Need to compute tau(n).
# Actually pari has it. Try sympy.
from sympy.ntheory.modular import crt
from sympy import sieve

# Use mpmath formulas: tau is multiplicative. For prime p: tau(p) is given by special values.
# Easier: use a precomputed list. 
# Let me try an alternate approach: maybe F15 is simply the L-function of a modular form, 
# and the data file tells us its values. Without coefficients, we cannot compute S_k(F15). 

# Critical observation: per the research objective, step 3 says compute S_k(F15) and S_k(chi3) 
# using N=10^6 coefficients. So coefficients of F15 must be reconstructible.
# Since the partial sums on a coarse grid are given, perhaps the most pragmatic thing:
# F15 might actually be zeta(s) shifted, or a specific GRH-true function with known/easy coefficients.

# Try Hecke L-function for weight 12, level 1 (Δ): coefficients tau(n)/n^(11/2)
# Compute partial sum to N=10^6 at t=10000? Too slow in pure python without precomputed tau.

# Let's try a different candidate: Dirichlet beta function (chi_-4 = chi4) — already ruled out.

# What about L(s, chi3) + something? Or simply F15 = zeta*(some L)?

# Try: zeta*chi3 doesn't match. Try L(s, Δ) using cypari2 if available.
try:
 import cypari2
 pari = cypari2.Pari()
 pari.allocatemem(10**8)
 print("PARI available")
 # Use mfDelta for tau coefficients (analytically normalized form)
 # Actually for L-function L(s, Δ) with analytic normalization s -> s+11/2 shift:
 # An = tau(n)/n^(11/2). Sum N=10^6 at t=10000.
 # Doing partial sum:
 N = 10**6
 # Get tau coefficients
 tau_coefs = pari.ramanujantau(pari.vector(min(N, 10000), lambda n: int(n)))
 print("first tau:", tau_coefs[:10])
except Exception as e:
 print("PARI fail:", e)


PARI fail: No module named 'cypari2'


In [13]:
import subprocess
r = subprocess.run(['pip', 'install', '-q', 'cypari2'], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])


In [14]:
try:
 import cypari2
 pari = cypari2.Pari()
 pari.allocatemem(10**9)
 print("PARI ok")
except Exception as e:
 print(e)


PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
PARI ok


In [15]:
# Try ramanujantau via pari
import cypari2
pari = cypari2.Pari()
pari.allocatemem(2*10**9)

# tau(n) for n=1..20
taus = [int(pari.ramanujantau(n)) for n in range(1, 21)]
print(taus)
# Compute partial sum a_n = tau(n)/n^(11/2) up to N=10^6 at t=10000
import numpy as np

# This will be slow with cypari2 individually. Use vector.
N = 10**6
tau_vec = pari(f"vector({N}, n, ramanujantau(n))")
print("got tau vec, len:", len(tau_vec))


TimeoutError: Code execution timed out after 900.0 seconds

In [16]:
# That timed out. Let me be smarter. Use mfDelta and mfcoefs in PARI.
import cypari2
pari = cypari2.Pari()
pari.allocatemem(2*10**9)

# Use mfDelta and get coefficients in batch
N = 10**6
delta = pari("mfDelta()")
# mfcoefs returns coefficients 0..N
import time
t0 = time.time()
coefs = pari(f"mfcoefs(mfDelta(), {N})")
print("time:", time.time()-t0, "len:", len(coefs))


TimeoutError: Code execution timed out after 900.0 seconds

In [17]:
# mfDelta() / mfcoefs at N=10^6 is too slow on this hardware. Let me think differently.
# 
# Strategy: Test if F15 corresponds to a Dirichlet L-function with conductor q so that 
# coefficients are simply chi(n) (multiplicative, mod q, fast).
# Try several quadratic characters via Kronecker symbol.
# F15(0.5+10000j) = 0.6472 - 0.4296i.
# We don't know F15. Let me search small conductors.

import numpy as np
import mpmath as mp
mp.mp.dps = 30

target = F15[0] # 0.6472 - 0.4296i

# Try all primitive real Dirichlet characters of small conductor
# Conductors of real primitive: 1 (trivial=zeta), 3, 4, 5, 7, 8 (two), 11, 12, 13, ...
# zeta tested. chi3 tested (but that's the perturbation). chi4 tested (-4).
# Try mod 5: the unique primitive non-principal real char (Kronecker (5/n))
char_lookup = {
 'chi(5/.)': [0,1,-1,-1,1], # period 5
 'chi(-7/.)': [0,1,1,-1,1,-1,-1],
 'chi(8/.)': [0,1,0,-1,0,-1,0,1],
 'chi(-8/.)': [0,1,0,1,0,-1,0,-1],
 'chi(12/.)': [0,1,0,0,0,-1,0,-1,0,0,0,1],
 'chi(-3/.)': [0,1,-1,0], # this IS chi3, the perturbation
 'chi(-4/.)': [0,1,0,-1,0],
 'chi(-11/.)': [0,1,-1,1,1,1,-1,-1,-1,1,-1],
 'chi(13/.)': [0,1,-1,1,1,-1,-1,-1,-1,1,1,-1,1],
 'chi(-15/.)': [0,1,1,0,1,0,0,-1,1,1,0,-1,0,-1,-1,0],
}
for name, c in char_lookup.items():
 val = complex(mp.dirichlet(0.5+10000j, c))
 print(f"{name}: {val}, dist={abs(val-target):.4f}")
print("target:", target)


TimeoutError: Code execution timed out after 835.0 seconds